## why are we doing deeplearning.
As of now, the model that is running in the prototype(RF model), is trained on the UCI dataset. The model is limitted to features (magnitude mean, std, min, max,range) meaning that its only being trained on how much they shake. So if 2 people shake for different reasons, the model gets confused. New features that will be developed for the model is rhythm(freq features), wrist angles(pitch and roll):

**Rhythm (frequency features)**: Walking is a regular beat of about 2 steps per second. Washing dishes or fidgeting is irregular. An FFT turns the signal into "how much energy at each speed of repetition," so walking shows a clear peak near 2 Hz and random movement doesn't.

**Wrist angle (pitch and roll)**: Gravity tells you which way the wrist is pointing. A wrist lying flat for a long time suggests lying down or sleeping. A wrist hanging at the side suggests standing. Your magnitude features throw this away because magnitude ignores direction.

right now the prototype model(RF model) is working in 2.56 windows and its like judging whether someone is walking from a 2-second clip: they might be pausing at a door. Thirty seconds shows the whole picture, and it matches how Capture-24 was labelled, since each label describes about half a minute.

To build a deep learning model, we are going to go with **Hidden Markov Model(HMM)** to smooth out the model and **1D CNN model** to classify:

the RF model judges in 2.56s windows which would give jumpy results, "still, still, WALK, still, still". but a real person rarely walks for 30 s in the middle of sitting. An HMM learns how likely each switch is, for example "still → still is very common, still → walking is rare." It then cleans up the sequence to: "still, still, still, still, still"

Think of it like autocorrect for activity labels: a single odd guess surrounded by consistent ones gets fixed. The Capture-24 authors say this step is "crucial," and it directly cuts false alerts

the 1d cnn model will be trained on capture 24 dataset due to size and features and ability to classify.

# Stage 0: prepare the data

Capture-24 (100 Hz wrist recordings, about a day per person) becomes 30 s windows at 50 Hz. **Nothing is trained here.**

1. Inspect the files
2. Functions: map labels, resample, cut windows
3. Test the functions on fake data with known answers
4. Run on all 151 people, one at a time, one `.npz` each (first tried on P001 to P010)
5. Check the saved files

**Kernel:** `anaconda3` (Python 3.11).

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt

In [2]:
DATA_DIR = Path('../../data/open datasets/capture24/capture24')
OUT_DIR = Path('processed')      # inside this folder, and git-ignored because it gets big

LABEL_SCHEME = 'label:Walmsley2020'
CLASSES = ['sleep', 'sedentary', 'light', 'moderate-vigorous']   # class number = position in this list
MISSING = -1                     # stands for "no label"

RAW_HZ = 100                     # Capture-24
TARGET_HZ = 50                   # our watch
WINDOW_S = 30
RAW_WIN = WINDOW_S * RAW_HZ      # 3000 samples before resampling
WIN = WINDOW_S * TARGET_HZ       # 1500 samples after resampling
MIN_PURITY = 0.90                # one label must cover at least this much of the window

RUN_ALL = True                   # WHY the switch: 10 people were tried first, so a mistake cost minutes and not an hour
OVERWRITE = False                # False: skip people already saved, so a stopped run can resume.
                                 # Set True after changing any processing code, or the old files are kept.

## 1. Inspect the files

Confirm the column names before writing a loader that depends on them.

In [3]:
all_files = sorted(DATA_DIR.glob('P*.csv.gz'))
print(len(all_files), 'participants:', all_files[0].name, '...', all_files[-1].name)

label_table = pd.read_csv(DATA_DIR / 'annotation-label-dictionary.csv')
print('\ndictionary columns:', list(label_table.columns))
print(label_table[LABEL_SCHEME].value_counts().to_string())

peek = pd.read_csv(all_files[0], nrows=5)
assert list(peek.columns) == ['time', 'x', 'y', 'z', 'annotation']
peek

151 participants: P001.csv.gz ... P151.csv.gz

dictionary columns: ['annotation', 'label:WillettsSpecific2018', 'label:WillettsMET2018', 'label:DohertySpecific2018', 'label:Willetts2018', 'label:Doherty2018', 'label:Walmsley2020']
label:Walmsley2020
light                94
sedentary            65
moderate-vigorous    46
sleep                 1


,time,x,y,z,annotation
0,2016-11-13 02:18:00.000000,-0.46669,-0.533341,0.658472,7030 sleeping;MET 0.95
1,2016-11-13 02:18:00.010000,-0.46669,-0.533341,0.658472,7030 sleeping;MET 0.95
2,2016-11-13 02:18:00.020000,-0.46669,-0.533341,0.658472,7030 sleeping;MET 0.95
3,2016-11-13 02:18:00.030000,-0.46669,-0.533341,0.658472,7030 sleeping;MET 0.95
4,2016-11-13 02:18:00.040000,-0.46669,-0.533341,0.658472,7030 sleeping;MET 0.95


## 2. Functions

Choices that matter:

- **Filter before resampling.** Keeping every 2nd sample turns movement faster than 25 Hz into fake slow movement (aliasing). A 20 Hz low-pass removes it first. It is the same filter as `capture24_windows.ipynb`, and it leaves a steady 1 g at exactly 1 g.
- **Labels are checked at 100 Hz**, before resampling, so no label is skipped over.
- **Each labelled stretch starts its own 30 s grid.** A stretch is labelled all the way through with no pause, so no window can contain a missing label. On the 10-person pilot this kept 6.2% more light and 4.3% more moderate-vigorous windows than one fixed grid, because short activities are not cut in awkward places.
- **A change of label does not end a stretch.** It is real continuous signal, and the 90% rule removes the mixed windows.
- **Filter the whole recording once, then cut.** The unlabelled parts are real signal too, so the filter has no artificial edges inside the recording.
- **The kept windows stay in time order with their start time**, because the HMM needs to know what came after what.

In [4]:
# one lookup: annotation text -> class number
label_of_text = label_table.set_index('annotation')[LABEL_SCHEME].map({name: i for i, name in enumerate(CLASSES)})
assert label_of_text.notna().all()     # every dictionary row must be one of our 4 classes

LOW_PASS = butter(4, 20, fs=RAW_HZ, output='sos')


def load_participant(pid):
    """Read one raw file -> times (nanoseconds), xyz (g), class number per sample."""
    # 'category' stores each long annotation text once, which saves a lot of RAM
    df = pd.read_csv(DATA_DIR / f'{pid}.csv.gz', dtype={'annotation': 'category'}, parse_dates=['time'])

    times = df['time'].to_numpy('datetime64[ns]').astype('int64')
    xyz = df[['x', 'y', 'z']].to_numpy()
    # WHY: one NaN would spread through the filter and ruin the whole recording
    assert not np.isnan(xyz).any(), f'{pid}: NaN in x/y/z'

    texts = df['annotation'].cat.categories
    # WHY: stop on unknown text, instead of quietly treating it as unlabelled
    assert texts.isin(label_of_text.index).all(), f'{pid}: annotation not in the dictionary'
    labels = df['annotation'].astype(object).map(label_of_text).fillna(MISSING).to_numpy('int8')
    return times, xyz, labels


def low_pass(xyz):
    """Remove movement faster than 20 Hz, so that keeping every 2nd sample is safe."""
    # WHY filtfilt: it runs forwards and backwards, so the signal is not delayed against its labels
    return sosfiltfilt(LOW_PASS, xyz, axis=0).astype('float32')


def find_stretches(times, labels):
    """(start, end) of every stretch that is labelled throughout and has no pause in the recording."""
    step = np.diff(times)
    pause = (step > 1.5 * 1e9 / RAW_HZ) | (step <= 0)     # more than 1.5 sample gaps, or time going backwards
    labelled = labels != MISSING
    breaks = np.flatnonzero(pause | (labelled[1:] != labelled[:-1])) + 1
    edges = np.concatenate([[0], breaks, [len(labels)]])
    return [(s, e) for s, e in zip(edges[:-1], edges[1:]) if labelled[s]]


def make_windows(times, xyz, labels):
    """One recording -> X (n, 1500, 3), y (n,), start times (n,), and a count of what was dropped."""
    smooth = low_pass(xyz)
    X, y, start = [], [], []
    stretches = find_stretches(times, labels)
    n_cut = 0
    for s, e in stretches:
        n = (e - s) // RAW_WIN                     # whole windows only; a leftover tail under 30 s is ignored
        if n == 0:
            continue
        n_cut += n
        e = s + n * RAW_WIN
        win_labels = labels[s:e].reshape(n, RAW_WIN)
        # how many samples of each class are in each window -> biggest class and its share
        counts = np.stack([(win_labels == k).sum(axis=1) for k in range(len(CLASSES))], axis=1)
        keep = counts.max(axis=1) >= MIN_PURITY * RAW_WIN

        X.append(smooth[s:e:RAW_HZ // TARGET_HZ].reshape(n, WIN, 3)[keep])   # every 2nd sample = 50 Hz
        y.append(counts.argmax(axis=1)[keep].astype('int8'))
        start.append(times[s:e:RAW_WIN][keep])

    if not X:                                      # nobody should end up empty, but fail clearly if they do
        raise ValueError('no usable windows')
    y = np.concatenate(y)
    info = {'labelled_h': round(float((labels != MISSING).sum()) / RAW_HZ / 3600, 2), 'stretches': len(stretches),
            'pauses': int((np.diff(times) > 1.5 * 1e9 / RAW_HZ).sum()), 'cut': n_cut, 'mixed': n_cut - len(y)}
    return np.concatenate(X), y, np.concatenate(start), info


def to_binary(y):
    """4 classes -> 0 = still (sleep, sedentary), 1 = moving (light, moderate-vigorous)."""
    return (y >= 2).astype('int8')


def process_participant(pid):
    """Load, window and save ONE person. Returns a summary row; the big arrays are freed on return."""
    t0 = time.time()
    X, y, start, info = make_windows(*load_participant(pid))
    OUT_DIR.mkdir(exist_ok=True)
    tmp = OUT_DIR / f'tmp_{pid}.npz'               # WHY: a crash mid-save can't leave a half file that looks finished
    np.savez(tmp, X=X, y4=y, y2=to_binary(y), start=start)
    tmp.replace(OUT_DIR / f'{pid}.npz')
    return {'pid': pid, **info, 'kept': len(y), 'seconds': round(time.time() - t0)}

## 3. Test the functions on fake data

Fake recordings have known answers, so a bug shows up here instead of hiding inside the real data.

In [5]:
def fake_recording(n_windows):
    """A perfectly still watch: 1 g on z, every sample labelled class 1 (sedentary)."""
    n = n_windows * RAW_WIN
    times = np.arange(n, dtype='int64') * 10_000_000   # 10 ms steps = 100 Hz
    return times, np.tile([0.0, 0.0, 1.0], (n, 1)), np.ones(n, 'int8')

# 1. shape is right, and a steady 1 g stays 1 g
t, xyz, lab = fake_recording(3)
X, y, start, info = make_windows(t, xyz, lab)
assert X.shape == (3, 1500, 3) and X.dtype == np.float32 and (y == 1).all() and np.allclose(X[..., 2], 1, atol=1e-5)
assert np.allclose(np.diff(start) / 1e9, 30)       # windows start 30 s apart, no overlap

# 2. a leftover tail shorter than 30 s is ignored
assert len(make_windows(t[:-1], xyz[:-1], lab[:-1])[0]) == 2

# 3. ONE unlabelled sample ends the stretch, and the next window starts right after it.
#    The slow ramp on x proves that signal, label and time all come from the same place.
lab3, ramp = lab.copy(), xyz.copy()
ramp[:, 0] = np.linspace(-1, 1, len(t))
lab3[RAW_WIN + 4] = MISSING; lab3[RAW_WIN + 5:] = 2
X, y, start, info = make_windows(t, ramp, lab3)
assert len(X) == 2 and info['stretches'] == 2 and list(y) == [1, 2]
assert start[1] == t[RAW_WIN + 5] and abs(X[1, 0, 0] - ramp[RAW_WIN + 5, 0]) < 1e-4
assert abs(X[1, -1, 0] - ramp[RAW_WIN + 5 + RAW_WIN - 2, 0]) < 1e-4

# 4. exactly 90% one label is kept, just under 90% is dropped, and the kept label is the bigger one
lab4 = lab.copy()
lab4[:300] = 2                                     # window 0: 10% class 2  -> keep as class 1
lab4[RAW_WIN:RAW_WIN + 301] = 2                    # window 1: 10.03% class 2 -> drop
X, y, start, info = make_windows(t, xyz, lab4)
assert len(X) == 2 and info['mixed'] == 1 and list(y) == [1, 1]

# 5. a pause in the recording ends the stretch, so no window joins two separate moments
t5 = t.copy(); t5[RAW_WIN + 50:] += 3_600_000_000_000
X, y, start, info = make_windows(t5, xyz, lab)
assert len(X) == 2 and info['pauses'] == 1 and start[1] == t5[RAW_WIN + 50]

# 6. 5 Hz movement survives, 40 Hz vibration is removed instead of turning into fake 10 Hz movement
def wobble_std(freq):
    t, xyz, lab = fake_recording(3)
    xyz[:, 2] += 0.5 * np.sin(2 * np.pi * freq * np.arange(len(t)) / RAW_HZ)
    return make_windows(t, xyz, lab)[0][1, :, 2].std()
print(f'5 Hz: {wobble_std(5):.3f} (perfect = 0.354), 40 Hz: {wobble_std(40):.5f} (perfect = 0)')
assert abs(wobble_std(5) - 0.354) < 0.01 and wobble_std(40) < 0.01

# 7. binary mapping
assert list(to_binary(np.array([0, 1, 2, 3]))) == [0, 0, 1, 1]
print('all checks passed')

5 Hz: 0.354 (perfect = 0.354), 40 Hz: 0.00000 (perfect = 0)
all checks passed


## 4. Run on everyone

About 40 minutes the first time. Stop it whenever you like and run it again: people already saved are skipped. The pilot (P001 to P010) contained nobody from the earlier 11-person test list.

In [6]:
all_pids = [f.name[:4] for f in all_files]
pids = all_pids if RUN_ALL else all_pids[:10]
SUMMARY = OUT_DIR / 'summary.csv'                  # WHY: the counts survive a restart, even when people are skipped

for i, pid in enumerate(pids, 1):
    if (OUT_DIR / f'{pid}.npz').exists() and not OVERWRITE:
        continue
    row = process_participant(pid)                 # only one person is in RAM at a time
    pd.DataFrame([row]).to_csv(SUMMARY, mode='a', header=not SUMMARY.exists(), index=False)
    print(f'[{i}/{len(pids)}]', row)

summary = pd.read_csv(SUMMARY).drop_duplicates('pid', keep='last').sort_values('pid')
assert set(pids) <= set(summary.pid)
print(f'{len(summary)} people saved, {summary.seconds.sum() / 60:.0f} min of processing')
print(summary[['labelled_h', 'stretches', 'pauses', 'cut', 'mixed', 'kept']].sum().round(1).to_string())
summary.describe().round(1)

[1/151] {'pid': 'P001', 'labelled_h': 20.75, 'stretches': 130, 'pauses': 0, 'cut': 2427, 'mixed': 0, 'kept': 2427, 'seconds': 18}
[2/151] {'pid': 'P002', 'labelled_h': 15.66, 'stretches': 59, 'pauses': 0, 'cut': 1852, 'mixed': 0, 'kept': 1852, 'seconds': 15}
[3/151] {'pid': 'P003', 'labelled_h': 16.44, 'stretches': 104, 'pauses': 0, 'cut': 1924, 'mixed': 0, 'kept': 1924, 'seconds': 17}
[4/151] {'pid': 'P004', 'labelled_h': 15.6, 'stretches': 75, 'pauses': 0, 'cut': 1834, 'mixed': 0, 'kept': 1834, 'seconds': 13}
[5/151] {'pid': 'P005', 'labelled_h': 19.29, 'stretches': 114, 'pauses': 0, 'cut': 2266, 'mixed': 0, 'kept': 2266, 'seconds': 17}
[6/151] {'pid': 'P006', 'labelled_h': 22.71, 'stretches': 38, 'pauses': 0, 'cut': 2720, 'mixed': 0, 'kept': 2720, 'seconds': 18}
[7/151] {'pid': 'P007', 'labelled_h': 21.08, 'stretches': 135, 'pauses': 0, 'cut': 2466, 'mixed': 0, 'kept': 2466, 'seconds': 17}
[8/151] {'pid': 'P008', 'labelled_h': 18.42, 'stretches': 112, 'pauses': 1, 'cut': 2159, 'mixe

,labelled_h,stretches,pauses,cut,mixed,kept,seconds
count,151.0,151.0,151.0,151.0,151.0,151.0,151.0
mean,17.0,87.7,0.1,1994.5,0.0,1994.5,15.4
std,3.3,30.5,0.2,389.0,0.1,389.0,1.8
min,6.1,25.0,0.0,719.0,0.0,719.0,8.0
25%,15.0,62.0,0.0,1774.5,0.0,1774.5,14.0
50%,17.1,87.0,0.0,2005.0,0.0,2005.0,16.0
75%,19.2,111.0,0.0,2250.0,0.0,2250.0,17.0
max,24.7,173.0,1.0,2914.0,1.0,2914.0,20.0


## 5. Check the saved files

Reads the files back from disk, so this tests what was really saved.

In [7]:
rows = []
for pid in pids:
    with np.load(OUT_DIR / f'{pid}.npz') as f:
        X, y4, y2, start = f['X'], f['y4'], f['y2'], f['start']
    assert X.shape[1:] == (1500, 3) and X.dtype == np.float32 and np.isfinite(X).all()
    assert len(X) == len(y4) == len(y2) == len(start)
    assert (y2 == to_binary(y4)).all()
    assert (np.diff(start) >= 30 * 10**9).all()    # in time order and never overlapping
    mag = np.sqrt((X ** 2).sum(axis=2))
    rows.append(pd.DataFrame({'pid': pid, 'y4': y4, 'y2': y2, 'mag_mean': mag.mean(axis=1), 'mag_std': mag.std(axis=1),
                              'next_is_adjacent': np.append(np.diff(start) == 30 * 10**9, False)}))
saved = pd.concat(rows, ignore_index=True)

print(f'{len(saved):,} windows ({len(saved) * 30 / 3600:.0f} hours) from {saved.pid.nunique()} people')
size_gb = sum((OUT_DIR / f'{p}.npz').stat().st_size for p in pids) / 1e9
print(f'on disk: {size_gb:.2f} GB')
assert len(saved) == summary.set_index('pid').loc[pids, 'kept'].sum()   # nothing lost between memory and disk
print(f'labelled time kept: {100 * len(saved) * 30 / 3600 / summary.set_index("pid").loc[pids, "labelled_h"].sum():.1f}%')
print(f'windows directly followed by another kept window: {100 * saved.next_is_adjacent.mean():.1f}%')

counts = saved.y4.value_counts().reindex(range(4), fill_value=0)
print('\nper class (a quieter class should have a smaller mag_std, and rest should sit near 1 g):')
print(pd.DataFrame({'class': CLASSES, 'windows': counts, 'percent': (100 * counts / len(saved)).round(1),
                    'mag_mean': saved.groupby('y4').mag_mean.mean().round(3),
                    'mag_std': saved.groupby('y4').mag_std.mean().round(3)}).to_string(index=False))
print(f'\nbinary: still {100 * (saved.y2 == 0).mean():.1f}%, moving {100 * (saved.y2 == 1).mean():.1f}%')

per_person = pd.crosstab(saved.pid, saved.y4).reindex(columns=range(4), fill_value=0).set_axis(CLASSES, axis=1)
print('\nwindows per person (some people have almost no moderate-vigorous):')
print(per_person.describe().loc[['min', '25%', '50%', '75%', 'max']].astype(int).to_string())
print('people with fewer than 10 moderate-vigorous windows:', int((per_person['moderate-vigorous'] < 10).sum()))

# for the HMM: where do label changes happen?
same_person = (saved.pid == saved.pid.shift(-1))
change = same_person & (saved.y4 != saved.y4.shift(-1))
print(f'\nlabel changes: {change.sum():,}, of which back-to-back (no gap): {(change & saved.next_is_adjacent).sum():,}')

301,164 windows (2510 hours) from 151 people
on disk: 5.42 GB
labelled time kept: 98.0%
windows directly followed by another kept window: 95.7%

per class (a quieter class should have a smaller mag_std, and rest should sit near 1 g):
            class  windows  percent  mag_mean  mag_std
            sleep   113359     37.6     0.999    0.009
        sedentary   119284     39.6     1.003    0.061
            light    54121     18.0     1.036    0.180
moderate-vigorous    14400      4.8     1.091    0.309

binary: still 77.2%, moving 22.8%

windows per person (some people have almost no moderate-vigorous):
     sleep  sedentary  light  moderate-vigorous
min      0        151     35                  0
25%    570        613    196                 24
50%    740        781    311                 59
75%    907        991    503                130
max   1600       1623    992                586
people with fewer than 10 moderate-vigorous windows: 28

label changes: 6,019, of which back-to-back